# Build an MCP Server

**What you will build:** your own **MCP server** with FastMCP, exercise its tools in the notebook, and see how any agent — PydanticAI, LangGraph, even Claude Desktop — could consume it. In 18 you *used* someone's MCP server; here you *publish* one. This is the payoff of the whole MCP idea: **write a tool once, use it from everywhere.**

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/18b_build_an_mcp_server.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
!pip install -q "fastmcp>=2.0,<3.0"

## Why publish a tool over MCP

In 1.2 a tool was a Python function bound to one agent. If a second agent (or a colleague, or Claude Desktop) needs the same capability, they'd re-implement it. An **MCP server** exposes your tool over a standard protocol, so *any* MCP-aware client can call it — no re-implementation, one place to maintain. FastMCP makes a server about as short as the function itself.

## Define a server

`FastMCP("name")` creates the server; `@mcp.tool` publishes a function. Just like PydanticAI, the **type hints become the schema and the docstring becomes the description** the model reads.

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("Demo Tools")

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b

@mcp.tool
def shout(text: str) -> str:
    """Return the text in upper case with an exclamation mark."""
    return f"{text.upper()}!"

print("server defined with tools: add, shout")

## Exercise it in-memory (no server process needed)

FastMCP's `Client` can connect **directly to the server object** — no network, no separate process — which is perfect for a notebook. This is exactly what a real client does over HTTP, just short-circuited. Read the returned value from `result.data`.

In [ ]:
from fastmcp import Client

async with Client(mcp) as client:                     # in-memory transport
    tools = await client.list_tools()
    print("tools the server exposes:")
    for t in tools:
        print(f"  {t.name} — {t.description}")

    result = await client.call_tool("add", {"a": 2, "b": 3})
    print("\nadd(2, 3) ->", result.data)             # .data is the deserialized Python value -> 5

That's a full MCP round-trip: list the tools, call one by name with arguments, read the result — the same protocol from 0.0, now spoken over MCP. (In a notebook use top-level `await`; never `asyncio.run()`, which errors inside the running loop.)

### The schema you didn't write

Same claim as 1.2's probe, other side of the protocol: the decorator generated a JSON schema from your type hints. Print it and compare with the ones you wrote by hand in 0.3:

In [ ]:
import json

async with Client(mcp) as client:
    tools = await client.list_tools()

for t in tools:
    print(t.name, "->", json.dumps(t.inputSchema))

`properties` from the type hints, `required` from the non-default arguments — byte-for-byte the shape you hand-authored in 0.3. Whether a tool lives in your process (1.2), on your MCP server (here) or on someone else's (18), the contract format is the same, which is the entire reason the ecosystem interoperates.

### What an error looks like through the protocol

Your tools *will* raise. Better to meet the failure shape here, deliberately, than in a production log: add a tool that can fail and call it wrongly:

In [ ]:
@mcp.tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

from fastmcp.exceptions import ToolError

async with Client(mcp) as client:
    print("ok:   ", (await client.call_tool("divide", {"a": 10, "b": 4})).data)
    try:
        await client.call_tool("divide", {"a": 1, "b": 0})
    except ToolError as e:
        print("error:", e)

The exception didn't cross the wire raw — MCP wrapped it into a protocol-level error (`ToolError` on the client side, carrying the server's message). When an *agent* is the client, PydanticAI surfaces this to the model as a failed tool call, and everything you know applies: the model can react, retry differently, or report it (0.3's error-as-observation, again). If you'd rather the model see a friendlier message, catch the exception inside the tool and return a string — the same choice you made in `run_agent`.

## Run it as a real HTTP server

To let *other* processes reach it, run the server over HTTP. `mcp.run()` blocks the kernel, so we write it to a file and run it in a terminal (not inline in Colab).

In [ ]:
%%writefile server.py
from fastmcp import FastMCP

mcp = FastMCP("Demo Tools")

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8000)   # serves at http://127.0.0.1:8000/mcp

Run it in a terminal:

```bash
pip install "fastmcp>=2.0,<3.0"
python server.py        # or: fastmcp run server.py:mcp --transport http --port 8000
```

```{note}
In Colab, `mcp.run()` blocks the single kernel and the port isn't public — so run the server locally or on a host, exactly like the FastAPI service in 30. The in-memory `Client(mcp)` cell above is how you test a server *inside* a notebook.
```

## Consume it from an agent — end to end, right here

Here's the part most tutorials leave as an exercise: your own server and a real agent, talking MCP, **in this cell**. `MCPToolset` accepts the in-memory `FastMCP` object directly — no terminal, no HTTP, same protocol (verified against pydantic-ai 2.5 + fastmcp 2.14):

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset

server_tools = MCPToolset(mcp)            # your in-memory server from above, as a toolset

consumer = Agent(model, toolsets=[server_tools],
                 system_prompt="You are a helpful assistant. Use tools when they help.")

async with consumer:
    result = await consumer.run("Use the tools: what is 7 plus 5, shouted?")
print(result.output)
print("calls:", [p.tool_name for m in result.all_messages()
                 for p in m.parts if p.part_kind == "tool-call"])

Full circle: a model chose tools it discovered over MCP, from a server you wrote four cells ago, and the loop underneath is still 0.3's. Over HTTP it's the same one-line change as in 18 — point the toolset at the URL instead of the object:

```python
agent = Agent(model, toolsets=[MCPToolset("http://127.0.0.1:8000/mcp")])
```

(That's what `server.py` above is for; run it in a terminal when you want the networked version.) The same URL works from a LangGraph agent (via `langchain-mcp-adapters`, 18) or from Claude Desktop's config. One server, many consumers — that's the point.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a real tool.** Publish `live_weather` from 1.2 (the `httpx` + wttr.in one) as an `@mcp.tool`, then ask the `consumer` agent about the weather. **Done when:** the tool-call list shows `live_weather` and the answer has today's real conditions.
2. **Friendly errors.** Change `divide` to catch the `ZeroDivisionError` itself and return the string `"Cannot divide by zero — ask the user for a different divisor."`. **Done when:** the agent, asked to divide by zero, explains the problem instead of surfacing a protocol error.
3. **The networked version.** Run `server.py` in a terminal and switch `server_tools` to the HTTP URL. **Done when:** the same `consumer.run(...)` works unchanged — proof that transport is an implementation detail.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Real tool on the server:**

```python
import httpx

@mcp.tool
def live_weather(city: str) -> str:
    """Get the CURRENT real weather for a city, e.g. 'Bilbao'."""
    try:
        r = httpx.get(f"https://wttr.in/{city}?format=j1", timeout=10)
        cur = r.json()["current_condition"][0]
        return f"{cur['temp_C']}C, {cur['weatherDesc'][0]['value']}"
    except Exception as e:
        return f"Weather service unavailable: {e}"

async with consumer:
    r = await consumer.run("What's the weather in Bilbao right now?")
```

Note you didn't touch the agent — the toolset re-lists the server's tools on connect. Adding a capability to every consumer of your server is now a server-side deploy, which is the operational win of MCP.

**2 — Friendly errors:** wrap the body in `try/except ZeroDivisionError: return "Cannot divide by zero — ..."`. The trade-off you just chose: a *string* result keeps the model in its comfort zone (it reads and reacts), while a raised error is machine-distinguishable for *code* consumers. APIs for models prefer the former; APIs for programs the latter.

**3 —** The only diff is `MCPToolset("http://127.0.0.1:8000/mcp")`. Everything downstream — schemas, calls, errors — is identical, because MCP specified the *protocol* and left the *transport* pluggable.
::::

**What's next:** you can now both consume (18) and publish (here) tools over MCP. In **19** we build the habit of debugging agents — the last skill of Block 1.